# Scenario Analysis with Fan Charts

Fan charts are a powerful tool for **visualizing forecast uncertainty**. Popularized
by the Bank of England's Inflation Report, they display the predictive distribution
as nested colored bands around the central forecast, with intensity decreasing from
center to tails.

**Topics covered:**
1. Building named scenarios (baseline, optimistic, pessimistic)
2. Fan charts from Monte Carlo simulation
3. Density forecasts (histograms and KDE)
4. Probability of macroeconomic events
5. Multi-variable fan charts

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

from forecastbox.scenarios import (
    ConditionalForecast,
    FanChart,
    MonteCarlo,
    ScenarioBuilder,
    SimpleVAR,
)

import sys
sys.path.insert(0, "..")
from utils.helpers import load_us_macro_quarterly, load_macro_brazil

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

## 1. Building Scenarios

We define three macroeconomic scenarios for the US economy:

- **Baseline**: The model's unconditional forecast (no policy intervention)
- **Optimistic**: Strong growth with falling unemployment and moderate rates
- **Pessimistic**: Stagflation-like scenario with high inflation and rising rates

In [ ]:
# Load data and estimate VAR
df = load_us_macro_quarterly()
var_names = ["gdp_growth", "inflation", "fed_funds", "unemployment"]
endog = df[var_names].values

model = SimpleVAR(endog, p_order=2, var_names=var_names)
steps = 8
print(f"VAR({model.p_order}) with {model.k_vars} variables, {endog.shape[0]} obs")

# Get unconditional forecast for baseline reference
cf = ConditionalForecast(model, method="analytic")
unc = cf.forecast(steps=steps, conditions=None, n_draws=1000, seed=42)

# Build 3 named scenarios
builder = ScenarioBuilder(model)

# Baseline: unconditional path for fed_funds
baseline_ff = unc["fed_funds"].point.tolist()
builder.add_scenario("baseline", {"fed_funds": baseline_ff},
                     description="Unconditional baseline")

# Optimistic: lower rates, economy growing
last_ff = df["fed_funds"].iloc[-1]
optimistic_ff = [max(last_ff - 0.25 * (h + 1), 0.0) for h in range(steps)]
builder.add_scenario("optimistic", {"fed_funds": optimistic_ff},
                     description="Rate cuts, strong growth")

# Pessimistic: rising rates + high inflation
pessimistic_ff = [last_ff + 0.50 * (h + 1) for h in range(steps)]
last_inf = df["inflation"].iloc[-1]
pessimistic_inf = [last_inf + 0.3 * (h + 1) for h in range(steps)]
builder.add_scenario("pessimistic",
                     {"fed_funds": pessimistic_ff, "inflation": pessimistic_inf},
                     description="Stagflation: rising rates + high inflation")

# Run all scenarios
scenario_results = builder.run(steps=steps, n_draws=1000, seed=42)

# Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
colors = {"baseline": "blue", "optimistic": "green", "pessimistic": "red"}
horizons = np.arange(1, steps + 1)

for idx, var in enumerate(var_names):
    ax = axes[idx // 2, idx % 2]
    for name in ["baseline", "optimistic", "pessimistic"]:
        fc = scenario_results.get(name, var)
        ax.plot(horizons, fc.point, "-o", color=colors[name],
                label=name.capitalize(), linewidth=2, markersize=4)
        if fc.lower_80 is not None:
            ax.fill_between(horizons, fc.lower_80, fc.upper_80,
                            alpha=0.1, color=colors[name])
    ax.set_title(var, fontsize=12)
    ax.set_xlabel("Horizon (quarters)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Three Macroeconomic Scenarios", fontsize=14)
fig.tight_layout()
plt.show()

## 2. Fan Charts

Fan charts display the full **predictive distribution** as nested probability bands.
We use Monte Carlo simulation (1000 draws) to generate future trajectories, then
extract quantiles to construct the fan chart.

The `MonteCarlo` class simulates N stochastic paths from the VAR model, and the
`FanChart` class visualizes them in the Bank of England style.

In [ ]:
# Monte Carlo simulation with 1000 paths
mc = MonteCarlo(model, n_paths=1000, seed=42, parametric=True)
paths = mc.simulate(steps=steps)
print(f"Simulated paths shape: {paths.shape}")
print(f"  -> {paths.shape[0]} paths x {paths.shape[1]} steps x {paths.shape[2]} variables")

# Build fan chart for GDP growth
history_gdp = df["gdp_growth"].values
fan_gdp = mc.fan_chart(variable="gdp_growth")

fig, ax = plt.subplots(figsize=(14, 6))
fan_gdp.plot(ax=ax, title="GDP Growth Fan Chart (1000 Monte Carlo Simulations)",
             color="steelblue", history_periods=20)
ax.set_ylabel("GDP Growth (%)")
ax.set_xlabel("Horizon")
plt.show()

# Summary statistics by horizon
stats = mc.statistics(variable="gdp_growth")
print("\nGDP Growth - Summary Statistics by Horizon:")
stats.round(3)

## 3. Density Forecasts

Beyond point forecasts and intervals, the Monte Carlo draws give us the **full
predictive density** at each forecast horizon. We can visualize these as histograms
or kernel density estimates (KDE).

Let's examine the forecast distribution at horizons t+4 (1 year ahead) and t+8 (2 years ahead).

In [ ]:
# Extract draws for GDP growth at t+4 and t+8
gdp_idx = var_names.index("gdp_growth")
draws_t4 = paths[:, 3, gdp_idx]  # horizon 4 (0-indexed: 3)
draws_t8 = paths[:, 7, gdp_idx]  # horizon 8 (0-indexed: 7)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, draws, h_label in [(axes[0], draws_t4, "t+4 (1 year)"),
                             (axes[1], draws_t8, "t+8 (2 years)")]:
    # Histogram
    ax.hist(draws, bins=40, density=True, alpha=0.5, color="steelblue",
            edgecolor="white", label="Histogram")

    # KDE overlay
    kde = gaussian_kde(draws)
    x_range = np.linspace(draws.min() - 1, draws.max() + 1, 200)
    ax.plot(x_range, kde(x_range), "r-", linewidth=2, label="KDE")

    # Mark median and mean
    ax.axvline(np.median(draws), color="navy", linestyle="--", linewidth=1.5,
               label=f"Median: {np.median(draws):.2f}")
    ax.axvline(np.mean(draws), color="darkred", linestyle=":", linewidth=1.5,
               label=f"Mean: {np.mean(draws):.2f}")

    ax.set_title(f"GDP Growth Density at {h_label}", fontsize=12)
    ax.set_xlabel("GDP Growth (%)")
    ax.set_ylabel("Density")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Density Forecasts: GDP Growth", fontsize=14)
fig.tight_layout()
plt.show()

# Also show density for inflation
inf_idx = var_names.index("inflation")
draws_inf_t4 = paths[:, 3, inf_idx]
draws_inf_t8 = paths[:, 7, inf_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, draws, h_label in [(axes[0], draws_inf_t4, "t+4 (1 year)"),
                             (axes[1], draws_inf_t8, "t+8 (2 years)")]:
    ax.hist(draws, bins=40, density=True, alpha=0.5, color="coral", edgecolor="white")
    kde = gaussian_kde(draws)
    x_range = np.linspace(draws.min() - 0.5, draws.max() + 0.5, 200)
    ax.plot(x_range, kde(x_range), "darkred", linewidth=2)
    ax.axvline(np.median(draws), color="navy", linestyle="--", linewidth=1.5,
               label=f"Median: {np.median(draws):.2f}")
    ax.set_title(f"Inflation Density at {h_label}", fontsize=12)
    ax.set_xlabel("Inflation (%)")
    ax.set_ylabel("Density")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Density Forecasts: Inflation", fontsize=14)
fig.tight_layout()
plt.show()

## 4. Probability of Events

The Monte Carlo draws allow us to compute the **probability of specific events**
by counting the fraction of simulated paths that satisfy a given condition.

Key events of interest:
- **Recession**: P(GDP growth < 0%) at each horizon
- **High inflation**: P(inflation > 4%) at each horizon
- **High unemployment**: P(unemployment > 8%) at each horizon

In [ ]:
# Probability of recession (GDP growth < 0%)
prob_recession = mc.probability(lambda y: y < 0.0, variable="gdp_growth")

# Probability of high inflation (> 4%)
prob_high_inf = mc.probability(lambda y: y > 4.0, variable="inflation")

# Probability of high unemployment (> 8%)
prob_high_unemp = mc.probability(lambda y: y > 8.0, variable="unemployment")

# Display probabilities
prob_df = pd.DataFrame({
    "P(GDP < 0%)": prob_recession,
    "P(Inflation > 4%)": prob_high_inf,
    "P(Unemployment > 8%)": prob_high_unemp,
}, index=[f"Q+{h+1}" for h in range(steps)])

print("Event Probabilities by Horizon:")
print(prob_df.round(3).to_string())

# Plot probabilities
fig, ax = plt.subplots(figsize=(12, 5))
horizons = np.arange(1, steps + 1)

ax.plot(horizons, prob_recession, "r-o", label="P(Recession: GDP < 0%)", linewidth=2, markersize=5)
ax.plot(horizons, prob_high_inf, "orange", marker="s", label="P(High Inflation > 4%)", linewidth=2, markersize=5)
ax.plot(horizons, prob_high_unemp, "purple", marker="^", label="P(High Unemployment > 8%)", linewidth=2, markersize=5)

ax.set_xlabel("Horizon (quarters)", fontsize=12)
ax.set_ylabel("Probability", fontsize=12)
ax.set_title("Probability of Adverse Events by Forecast Horizon", fontsize=14)
ax.set_ylim(-0.02, 1.02)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 5. Multi-Variable Fan Charts

In a VAR system, all variables are jointly determined. Let's visualize fan charts
for GDP growth, inflation, and unemployment side by side to see the co-movement
of uncertainty across variables.

In [ ]:
# Multi-variable fan charts: GDP, Inflation, Unemployment
plot_vars = ["gdp_growth", "inflation", "unemployment"]
colors_map = {"gdp_growth": "steelblue", "inflation": "coral", "unemployment": "mediumpurple"}
titles = {"gdp_growth": "GDP Growth (%)", "inflation": "Inflation (%)", "unemployment": "Unemployment (%)"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, var in enumerate(plot_vars):
    ax = axes[idx]
    fan = mc.fan_chart(variable=var)
    fan.plot(ax=ax, title=titles[var], color=colors_map[var], history_periods=0)
    ax.set_xlabel("Horizon (quarters)")

    # Add width annotation at final horizon
    width = fan.width_at_horizon(h=steps - 1, level=0.80)
    ax.annotate(f"80% width: {width:.2f}", xy=(0.95, 0.05), xycoords="axes fraction",
                ha="right", fontsize=9, style="italic",
                bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

fig.suptitle("Multi-Variable Fan Charts (1000 Monte Carlo Simulations)", fontsize=14)
fig.tight_layout()
plt.show()

# Fan chart data as DataFrame
print("\nFan Chart Data - GDP Growth:")
fan_gdp.to_dataframe().round(3)

## Exercise 1: Build fan chart for Brazilian GDP growth

Load the `macro_brazil.csv` dataset, estimate a VAR model, run Monte Carlo simulation,
and build a fan chart for Brazilian GDP growth (`pib_mensal`).

In [ ]:
# TODO: Exercise 1
# Hint:
# df_br = load_macro_brazil()
# endog_br = df_br[["ipca", "selic", "cambio", "pib_mensal", "producao_industrial"]].values
# model_br = SimpleVAR(endog_br, p_order=2, var_names=[...])
# mc_br = MonteCarlo(model_br, n_paths=1000, seed=42)
# mc_br.simulate(steps=12)
# fan_br = mc_br.fan_chart(variable="pib_mensal")
# fan_br.plot(title="Brazilian GDP Growth Fan Chart")

## Exercise 2: Calculate probability of stagflation scenario

Using the Monte Carlo simulation, calculate the **joint probability** of stagflation:
GDP growth < 0% AND inflation > 4% simultaneously. Plot how this probability evolves
over the forecast horizon.

In [ ]:
# TODO: Exercise 2
# Hint: Use the raw paths array to compute joint probabilities
# gdp_paths = paths[:, :, gdp_idx]
# inf_paths = paths[:, :, inf_idx]
# joint_mask = (gdp_paths < 0.0) & (inf_paths > 4.0)
# prob_stagflation = joint_mask.mean(axis=0)  # probability at each horizon